# Notebook 03 — EdoFarm AI Translation and Bilingual Dataset Construction

## 1. Introduction

This notebook develops the English–Edo (Bini) bilingual dataset for EdoFarm AI.

The goal is to prepare parallel agricultural questions and answers in English and Edo (Bini), allowing farmers to interact with the chatbot in either language.

This notebook focuses on translation data preparation, translation generation, translation review and bilingual dataset construction.

## 2. Translation approach

EdoFarm AI will maintain two independent translations for each translatable text:

1. Machine-generated translation.
2. Human translation.

A third field will contain the final selected translation.

The selection priority is:

- Use the human translation when it has been reviewed and approved.
- Otherwise, use the machine-generated translation when available.
- If neither is available, leave the final translation empty.

Human translations are preferred because they can incorporate natural Edo expressions, local terminology and cultural context.

Machine-generated translations are retained as a fallback and comparison source.

## 3. Translation scope

The working collection has two categories:

### Agricultural Q&A
Both the English question and answer will have machine and human translation fields.

### Out-of-scope classifier examples
Only the English question will be translated.

Their original answers will not be translated or used as agricultural knowledge.

## 4. Translation quality

Machine translations are drafts and must not automatically be treated as validated Edo.

Human translations must also undergo review before being selected as the preferred version.

Translation correctness and agricultural correctness will be tracked separately.

No machine or human translation will automatically make an unverified agricultural answer trustworthy.

## 5. Translation Architecture

Each original English record will have a unique translation_group_id.

The English, machine-generated Edo and human Edo versions of the same example will share this identifier.

### Question fields

- edo_question_machine
- edo_question_human
- edo_question_final

### Answer fields

- edo_answer_machine
- edo_answer_human
- edo_answer_final

### Translation selection

The final question and answer are selected independently.

A human translation marked reviewed_approved takes priority.

A human draft or translation needing revision does not override an available machine fallback.

The selected source is recorded so that the origin of every final translation is traceable.

### Translation review

Question and answer translation reviews are tracked separately.

For classifier-only examples, answer translation and answer translation review are marked not applicable.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


## 6. Locating Project Files

We will load the reviewed English dataset produced in Notebook 02.

We will also load the classifier candidate export to recover the near-duplicate group identifiers.

The original reviewed dataset will remain unchanged.

All translation preparation and exports will be performed using separate working copies.

In [2]:
# Locate the project root
PROJECT_ROOT = Path.cwd()

if (
    not (PROJECT_ROOT / "data").exists()
    and (PROJECT_ROOT.parent / "data").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Input files
reviewed_path = (
    METADATA_DIR / "edofarm_full_reviewed_dataset.csv"
)

classifier_path = (
    PROCESSED_DIR / "edofarm_training_dataset.csv"
)

# Confirm availability
for path in [reviewed_path, classifier_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print("Project root:", PROJECT_ROOT)
print("Reviewed dataset:", reviewed_path)
print("Classifier dataset:", classifier_path)
print("Output directory:", PROCESSED_DIR)

Project root: c:\Users\XXMUGENX\3D Objects\edofarm-ai
Reviewed dataset: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata\edofarm_full_reviewed_dataset.csv
Classifier dataset: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\edofarm_training_dataset.csv
Output directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed


## 7. Loading the English Source Dataset

Load the reviewed dataset and classifier candidate collection.

The reviewed dataset provides the original questions, answers and review decisions.

The classifier candidate collection provides the duplicate-group identifiers needed for future leakage prevention.

All identifiers and text fields will be loaded as strings to preserve their original values.

In [3]:
reviewed_df = pd.read_csv(
    reviewed_path,
    dtype=str,
    keep_default_na=False
)

classifier_df = pd.read_csv(
    classifier_path,
    dtype=str,
    keep_default_na=False
)

print("Reviewed dataset:", reviewed_df.shape)
print("Classifier candidates:", classifier_df.shape)

# Basic validation
assert len(reviewed_df) == 522
assert reviewed_df["record_id"].is_unique

assert len(classifier_df) == 517
assert classifier_df["record_id"].is_unique

assert "near_duplicate_group" in classifier_df.columns
assert "split_group_id" in classifier_df.columns

print("\nSource datasets loaded and validated.")

Reviewed dataset: (522, 29)
Classifier candidates: (517, 31)

Source datasets loaded and validated.


## 8. Selecting Translation Candidates

We will construct two independent collections.

### Agricultural Q&A collection

Select records satisfying all conditions:

- review_decision = accept
- answer_quality = clear_and_relevant
- primary_intent is not out_of_scope

Both questions and answers will be translated.

Their existing knowledge_base_status and source_check_status fields will be preserved. These records are still not automatically approved agricultural knowledge.

### Out-of-scope classifier collection

Select records satisfying:

- review_decision = accept
- primary_intent = out_of_scope
- classifier_label_status = provisional_pending_team_confirmation

Only their questions will be translated.

Their original answers will remain excluded from the translation collection.

These two collections are intended for different downstream uses and must remain distinguishable.

In [4]:
# Agricultural question-and-answer candidates
qa_mask = (
    (reviewed_df["review_decision"] == "accept") &
    (reviewed_df["answer_quality"] == "clear_and_relevant") &
    (reviewed_df["primary_intent"] != "out_of_scope")
)

qa_candidates = reviewed_df[qa_mask].copy()

# Out-of-scope classifier questions
outscope_mask = (
    (reviewed_df["review_decision"] == "accept") &
    (reviewed_df["primary_intent"] == "out_of_scope") &
    (
        reviewed_df["classifier_label_status"]
        == "provisional_pending_team_confirmation"
    )
)

outscope_candidates = reviewed_df[outscope_mask].copy()

# Verify no overlap
assert set(qa_candidates["record_id"]).isdisjoint(
    set(outscope_candidates["record_id"])
)

print("AGRICULTURAL Q&A CANDIDATES:", len(qa_candidates))
print("OUT-OF-SCOPE QUESTIONS:", len(outscope_candidates))
print(
    "TOTAL:",
    len(qa_candidates) + len(outscope_candidates)
)

print("\nAgricultural intent distribution:")
display(qa_candidates["primary_intent"].value_counts())

print("\nOut-of-scope examples:")
display(
    outscope_candidates[
        ["record_id", "question", "primary_intent"]
    ].head(10)
)

assert len(qa_candidates) == 231
assert len(outscope_candidates) == 75

print("\nCandidate selection completed.")

AGRICULTURAL Q&A CANDIDATES: 231
OUT-OF-SCOPE QUESTIONS: 75
TOTAL: 306

Agricultural intent distribution:


primary_intent
soil_management           44
disease_management        40
fertilizer_application    40
harvesting                33
irrigation                32
pest_management           25
crop_cultivation          17
Name: count, dtype: int64


Out-of-scope examples:


,record_id,question,primary_intent
10,AQA_00113,Why do cows stand around in fields eating all ...,out_of_scope
11,AQA_00115,what is the use of cow's special stomach,out_of_scope
12,AQA_00117,How can cows make so much milk?,out_of_scope
13,AQA_00118,Why are pigs so dirty?,out_of_scope
14,AQA_00123,Do cows cause global warming?,out_of_scope
15,AQA_00136,what is animal welfare?,out_of_scope
16,AQA_00140,List the breeds of dairy cattle?,out_of_scope
24,AQA_00252,what is animal husbandry?,out_of_scope
31,AQA_00289,how can rainfall increase water quality.,out_of_scope
32,AQA_00302,define Livestock management practices?,out_of_scope



Candidate selection completed.


## 9. Constructing the Bilingual Translation Queue

We will create a separate translation queue containing the English source text and fields for both machine-generated and human translations.

Each record will have:

### Questions
- edo_question_machine
- edo_question_human
- edo_question_final

### Answers
- edo_answer_machine
- edo_answer_human
- edo_answer_final

### Translation tracking
- machine translation status
- human translation review status
- selected translation source
- translation notes

The final translation selection follows this priority:

1. Approved human translation.
2. Available machine-generated translation as fallback.
3. Empty translation if neither is available.

Out-of-scope classifier examples will have question translations only.

All translations remain subject to their respective quality checks.

In [5]:
# Retrieve duplicate-group metadata
group_metadata = classifier_df[
    [
        "record_id",
        "near_duplicate_group",
        "split_group_id"
    ]
].copy()

assert group_metadata["record_id"].is_unique

# Verify all selected records have duplicate metadata
selected_ids = (
    set(qa_candidates["record_id"]) |
    set(outscope_candidates["record_id"])
)

missing_ids = selected_ids - set(group_metadata["record_id"])

if missing_ids:
    raise ValueError(f"Missing duplicate metadata: {missing_ids}")


def build_translation_rows(source_df, unit_type):

    rows = []

    for _, row in source_df.iterrows():

        is_qa = unit_type == "bilingual_qa"

        rows.append({
            "translation_group_id": f"TRG_{row['record_id']}",
            "record_id": row["record_id"],

            "source_dataset": row["source_dataset"],
            "source_record_id": row["source_record_id"],

            "translation_unit_type": unit_type,
            "primary_intent": row["primary_intent"],

            "crop": row["crop"],
            "country": row["country"],
            "state_region": row["state_region"],
            "local_area": row["local_area"],

            # Original English content
            "english_question": row["question"],
            "english_answer": row["answer"] if is_qa else "",

            # Machine-generated translations
            "edo_question_machine": "",
            "edo_answer_machine": "",
            "machine_question_status": "not_started",
            "machine_answer_status": (
                "not_started" if is_qa
                else "not_applicable_classifier_only"
            ),

            # Human translations
            "edo_question_human": "",
            "edo_answer_human": "",

            "human_question_review_status": "not_started",
            "human_answer_review_status": (
                "not_started" if is_qa
                else "not_applicable_classifier_only"
            ),

            # Final selected translations
            "edo_question_final": "",
            "edo_answer_final": "",

            "question_translation_selected_from": "missing",
            "answer_translation_selected_from": (
                "missing" if is_qa
                else "not_applicable_classifier_only"
            ),

            # Translation metadata
            "translation_method": "not_started",
            "translator_id": "",
            "translator_variant": "Edo (Bini)",

            "translation_review_status": "not_started",
            "meaning_preservation_status": "not_assessed",

            # Preserve agricultural verification status
            "agricultural_accuracy": row["agricultural_accuracy"],
            "source_check_status": row["source_check_status"],
            "knowledge_base_status": row["knowledge_base_status"],

            "translation_notes": ""
        })

    return pd.DataFrame(rows)


# Construct both collections
qa_translation_df = build_translation_rows(
    qa_candidates,
    "bilingual_qa"
)

outscope_translation_df = build_translation_rows(
    outscope_candidates,
    "question_only_classifier"
)

# Combine
translation_queue = pd.concat(
    [qa_translation_df, outscope_translation_df],
    ignore_index=True
)

# Attach duplicate metadata
translation_queue = translation_queue.merge(
    group_metadata,
    on="record_id",
    how="left",
    validate="one_to_one"
)

print("TRANSLATION QUEUE CREATED")
print("--------------------------------")
print("Total records:", len(translation_queue))
print("Agricultural Q&A:", len(qa_translation_df))
print("Out-of-scope questions:", len(outscope_translation_df))
print(
    "Missing duplicate metadata:",
    translation_queue["split_group_id"].isna().sum()
)

display(translation_queue.head(5))

# Integrity checks
assert len(translation_queue) == 306
assert translation_queue["record_id"].is_unique
assert translation_queue["translation_group_id"].is_unique

assert translation_queue["edo_question_machine"].eq("").all()
assert translation_queue["edo_question_human"].eq("").all()
assert translation_queue["edo_question_final"].eq("").all()

assert translation_queue.loc[
    translation_queue["translation_unit_type"] == "question_only_classifier",
    "english_answer"
].eq("").all()

assert translation_queue.loc[
    translation_queue["translation_unit_type"] == "question_only_classifier",
    "edo_answer_machine"
].eq("").all()

assert translation_queue["split_group_id"].ne("").all()

print("\nAll translation queue checks passed.")

TRANSLATION QUEUE CREATED
--------------------------------
Total records: 306
Agricultural Q&A: 231
Out-of-scope questions: 75
Missing duplicate metadata: 0


,translation_group_id,record_id,source_dataset,source_record_id,translation_unit_type,primary_intent,crop,country,state_region,local_area,...,translator_id,translator_variant,translation_review_status,meaning_preservation_status,agricultural_accuracy,source_check_status,knowledge_base_status,translation_notes,near_duplicate_group,split_group_id
0,TRG_AQA_00009,AQA_00009,Agronomy QA Pairs,9,bilingual_qa,disease_management,,,,,...,,Edo (Bini),not_started,not_assessed,not_expert_verified,source_checked,candidate_pending_source_verification,,,REC_AQA_00009
1,TRG_AQA_00027,AQA_00027,Agronomy QA Pairs,27,bilingual_qa,pest_management,,,,,...,,Edo (Bini),not_started,not_assessed,not_expert_verified,not_checked,candidate_pending_source_verification,,,REC_AQA_00027
2,TRG_AQA_00031,AQA_00031,Agronomy QA Pairs,31,bilingual_qa,harvesting,,,,,...,,Edo (Bini),not_started,not_assessed,not_expert_verified,not_checked,candidate_pending_source_verification,,,REC_AQA_00031
3,TRG_AQA_00045,AQA_00045,Agronomy QA Pairs,45,bilingual_qa,crop_cultivation,,,,,...,,Edo (Bini),not_started,not_assessed,not_expert_verified,not_checked,candidate_pending_source_verification,,,REC_AQA_00045
4,TRG_AQA_00061,AQA_00061,Agronomy QA Pairs,61,bilingual_qa,pest_management,,,,,...,,Edo (Bini),not_started,not_assessed,not_expert_verified,not_checked,candidate_pending_source_verification,,,REC_AQA_00061



All translation queue checks passed.


## 10. Translation Selection Logic

The final Edo translation is selected independently for questions and answers.

Human translations have the highest priority, but only when marked `reviewed_approved`.

If the human translation is not approved, an available machine-generated translation is used as a fallback.

If neither version is available, the final translation remains empty.

The selected source is recorded as:

- human
- machine
- missing

For out-of-scope classifier records, answer translation is not applicable.

The selection function does not modify the original English text or either translation draft.

In [6]:
def select_translation(human_text, machine_text, human_status):

    human_text = str(human_text).strip()
    machine_text = str(machine_text).strip()
    human_status = str(human_status).strip()

    # Approved human translation takes priority
    if human_text and human_status == "reviewed_approved":
        return human_text, "human"

    # Machine translation is the fallback
    if machine_text:
        return machine_text, "machine"

    # No translation available
    return "", "missing"


def resolve_translations(df):

    df = df.copy()

    for index, row in df.iterrows():

        # Resolve question translation
        question, question_source = select_translation(
            row["edo_question_human"],
            row["edo_question_machine"],
            row["human_question_review_status"]
        )

        df.at[index, "edo_question_final"] = question
        df.at[index, "question_translation_selected_from"] = (
            question_source
        )

        # Resolve answer translation only for agricultural Q&A
        if row["translation_unit_type"] == "bilingual_qa":

            answer, answer_source = select_translation(
                row["edo_answer_human"],
                row["edo_answer_machine"],
                row["human_answer_review_status"]
            )

            df.at[index, "edo_answer_final"] = answer
            df.at[index, "answer_translation_selected_from"] = (
                answer_source
            )

    return df


print("Translation selection function defined.")

Translation selection function defined.


In [7]:
test_cases = [
    {
        "human": "Human version",
        "machine": "Machine version",
        "status": "reviewed_approved",
        "expected": "human"
    },
    {
        "human": "Human draft",
        "machine": "Machine version",
        "status": "pending_review",
        "expected": "machine"
    },
    {
        "human": "",
        "machine": "Machine version",
        "status": "not_started",
        "expected": "machine"
    },
    {
        "human": "",
        "machine": "",
        "status": "not_started",
        "expected": "missing"
    }
]

for case in test_cases:

    _, selected_source = select_translation(
        case["human"],
        case["machine"],
        case["status"]
    )

    assert selected_source == case["expected"]

print("All four translation-selection tests passed.")

All four translation-selection tests passed.


## 11. Exporting the Initial Translation Queue

We will save the prepared bilingual dataset to data/processed/.

The exported queue contains 306 records:
- 231 agricultural Q&A examples.
- 75 out-of-scope classifier questions.

At this stage, no Edo translation has been generated.

The machine, human and final translation fields remain empty.

We will also export a summary showing the number of records by translation type and agricultural intent.

In [8]:
# Output paths
translation_queue_path = (
    PROCESSED_DIR / "edofarm_translation_queue_v1.csv"
)

translation_summary_path = (
    PROCESSED_DIR / "edofarm_translation_summary_v1.csv"
)

# Build summary
translation_summary = (
    translation_queue.groupby(
        ["translation_unit_type", "primary_intent"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

# Export
translation_queue.to_csv(
    translation_queue_path,
    index=False,
    encoding="utf-8-sig"
)

translation_summary.to_csv(
    translation_summary_path,
    index=False,
    encoding="utf-8-sig"
)

# Read back for validation
exported_queue = pd.read_csv(
    translation_queue_path,
    dtype=str,
    keep_default_na=False
)

exported_summary = pd.read_csv(
    translation_summary_path,
    dtype=str,
    keep_default_na=False
)

# Integrity checks
assert len(exported_queue) == 306
assert exported_queue["record_id"].is_unique
assert exported_queue["translation_group_id"].is_unique

assert (
    exported_queue["record_id"].tolist()
    == translation_queue["record_id"].tolist()
)

# Confirm English source text was preserved
assert (
    exported_queue["english_question"].tolist()
    == translation_queue["english_question"].tolist()
)

assert (
    exported_queue["english_answer"].tolist()
    == translation_queue["english_answer"].tolist()
)

# Confirm no translation was added prematurely
for column in [
    "edo_question_machine",
    "edo_answer_machine",
    "edo_question_human",
    "edo_answer_human",
    "edo_question_final",
    "edo_answer_final"
]:
    assert exported_queue[column].eq("").all()

print("TRANSLATION EXPORTS COMPLETED")
print("--------------------------------")
print("Translation queue:", len(exported_queue))
print("Agricultural Q&A:",
      (exported_queue["translation_unit_type"] == "bilingual_qa").sum())
print("Out-of-scope questions:",
      (exported_queue["translation_unit_type"] == "question_only_classifier").sum())

print("\nFiles saved:")
print(translation_queue_path)
print(translation_summary_path)

print("\nTranslation summary:")
display(exported_summary)

TRANSLATION EXPORTS COMPLETED
--------------------------------
Translation queue: 306
Agricultural Q&A: 231
Out-of-scope questions: 75

Files saved:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\edofarm_translation_queue_v1.csv
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\edofarm_translation_summary_v1.csv

Translation summary:


,translation_unit_type,primary_intent,record_count
0,bilingual_qa,crop_cultivation,17
1,bilingual_qa,disease_management,40
2,bilingual_qa,fertilizer_application,40
3,bilingual_qa,harvesting,33
4,bilingual_qa,irrigation,32
5,bilingual_qa,pest_management,25
6,bilingual_qa,soil_management,44
7,question_only_classifier,out_of_scope,75


## 12. Preparing Machine Translation Batches

The translation queue will be divided into manageable batches of 10 records.

Two separate batch collections will be created:

1. Agricultural Q&A translation batches.
2. Out-of-scope classifier question batches.

Agricultural batches contain English questions and answers.

Out-of-scope batches contain English questions only.

Each batch preserves:
- translation_group_id
- record_id
- translation_unit_type
- primary_intent
- English source text
- duplicate-group metadata

The batches will be processed independently to reduce translation errors and make missing or altered records easier to identify.

The generated translations remain machine drafts pending human review.

In [9]:
BATCH_SIZE = 10

BATCH_ROOT = PROCESSED_DIR / "translation_batches"

QA_BATCH_DIR = BATCH_ROOT / "agricultural_qa"
OUTSCOPE_BATCH_DIR = BATCH_ROOT / "out_of_scope"

QA_BATCH_DIR.mkdir(parents=True, exist_ok=True)
OUTSCOPE_BATCH_DIR.mkdir(parents=True, exist_ok=True)

batch_manifest = []


def export_translation_batches(df, prefix, output_dir):

    batch_count = 0

    for start in range(0, len(df), BATCH_SIZE):

        batch_count += 1

        batch = df.iloc[
            start:start + BATCH_SIZE
        ].copy()

        batch_id = f"{prefix}_{batch_count:03d}"

        batch.insert(0, "batch_id", batch_id)

        output_path = (
            output_dir / f"{batch_id}_input.csv"
        )

        batch.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        batch_manifest.append({
            "batch_id": batch_id,
            "translation_unit_type": batch[
                "translation_unit_type"
            ].iloc[0],
            "record_count": len(batch),
            "file_name": output_path.name,
            "status": "not_started"
        })

        print(
            batch_id,
            "| Records:", len(batch),
            "| Saved:", output_path.name
        )

    return batch_count


qa_batch_count = export_translation_batches(
    translation_queue[
        translation_queue["translation_unit_type"]
        == "bilingual_qa"
    ].reset_index(drop=True),
    "QA",
    QA_BATCH_DIR
)

outscope_batch_count = export_translation_batches(
    translation_queue[
        translation_queue["translation_unit_type"]
        == "question_only_classifier"
    ].reset_index(drop=True),
    "OOS",
    OUTSCOPE_BATCH_DIR
)

batch_manifest_df = pd.DataFrame(batch_manifest)

manifest_path = (
    BATCH_ROOT / "translation_batch_manifest.csv"
)

batch_manifest_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

assert qa_batch_count == 24
assert outscope_batch_count == 8
assert len(batch_manifest_df) == 32
assert batch_manifest_df["record_count"].sum() == 306

print("\nBATCH GENERATION COMPLETED")
print("--------------------------------")
print("Agricultural Q&A batches:", qa_batch_count)
print("Out-of-scope batches:", outscope_batch_count)
print("Total batches:", len(batch_manifest_df))
print("Total records:", batch_manifest_df["record_count"].sum())
print("Manifest:", manifest_path)

QA_001 | Records: 10 | Saved: QA_001_input.csv
QA_002 | Records: 10 | Saved: QA_002_input.csv
QA_003 | Records: 10 | Saved: QA_003_input.csv
QA_004 | Records: 10 | Saved: QA_004_input.csv
QA_005 | Records: 10 | Saved: QA_005_input.csv
QA_006 | Records: 10 | Saved: QA_006_input.csv
QA_007 | Records: 10 | Saved: QA_007_input.csv
QA_008 | Records: 10 | Saved: QA_008_input.csv
QA_009 | Records: 10 | Saved: QA_009_input.csv
QA_010 | Records: 10 | Saved: QA_010_input.csv
QA_011 | Records: 10 | Saved: QA_011_input.csv
QA_012 | Records: 10 | Saved: QA_012_input.csv
QA_013 | Records: 10 | Saved: QA_013_input.csv
QA_014 | Records: 10 | Saved: QA_014_input.csv
QA_015 | Records: 10 | Saved: QA_015_input.csv
QA_016 | Records: 10 | Saved: QA_016_input.csv
QA_017 | Records: 10 | Saved: QA_017_input.csv
QA_018 | Records: 10 | Saved: QA_018_input.csv
QA_019 | Records: 10 | Saved: QA_019_input.csv
QA_020 | Records: 10 | Saved: QA_020_input.csv
QA_021 | Records: 10 | Saved: QA_021_input.csv
QA_022 | Reco

## 13. Machine Translation Instructions

Machine translation will generate provisional Edo (Bini) drafts.

The translator must preserve:
- Original meaning.
- Agricultural terminology.
- Crop and disease names.
- Numbers, quantities and measurements.
- Chemical names and warnings.
- Negations and conditional statements.

The translator must not introduce new advice, correct the agricultural content, or silently change the meaning.

Where an Edo equivalent is uncertain, the translator should retain the technical term where appropriate and document the uncertainty.

Only machine translation fields will be populated.

Human translation fields and final selected translation fields must remain untouched.

An uncertain translation must not automatically become the final selected translation.

## 14. Loading Machine-Generated Translation Results

The machine translation output has been merged into the original translation queue.

We will now load the merged dataset and verify:

- All 306 original records are present.
- Record IDs are unique.
- English source text remains available.
- Machine translation fields are populated where translations were generated.
- Human translation fields remain untouched.
- Out-of-scope records contain question translations only.

Machine translations are provisional fallbacks. They are not considered human-verified Edo translations.

In [11]:
machine_filled_path = (
    PROCESSED_DIR / "edofarm_translation_queue_v2_machine_filled.csv"
)

if not machine_filled_path.exists():
    raise FileNotFoundError(
        f"Machine translation file not found: {machine_filled_path}"
    )

machine_translation_df = pd.read_csv(
    machine_filled_path,
    dtype=str,
    keep_default_na=False
)

print("Dataset shape:", machine_translation_df.shape)

# Structural validation
assert len(machine_translation_df) == 306
assert machine_translation_df["record_id"].is_unique
assert machine_translation_df["translation_group_id"].is_unique

assert (
    machine_translation_df["translation_unit_type"]
    .value_counts().to_dict()
    ==
    {
        "bilingual_qa": 231,
        "question_only_classifier": 75
    }
)

# Translation coverage
assert machine_translation_df[
    "edo_question_machine"
].ne("").all()

# Human fields must remain untouched
assert machine_translation_df[
    "edo_question_human"
].eq("").all()

assert machine_translation_df[
    "edo_answer_human"
].eq("").all()

print("\nMachine translation dataset loaded successfully.")

Dataset shape: (306, 35)

Machine translation dataset loaded successfully.


## 15. Selecting Provisional Edo Translations

The final translation fields will be populated using the established selection rules.

Priority:

1. Approved human translation.
2. Available machine translation.
3. Missing translation, flagged for attention.

At this stage, human translations have not yet been supplied.

Therefore, available machine translations will serve as provisional fallbacks.

Translations marked uncertain or failed will not automatically be selected.

Question and answer selection are handled independently.

Out-of-scope records will not receive answer translations.

In [12]:
def select_translation_v2(
    human_text,
    machine_text,
    human_status,
    machine_status
):
    human_text = str(human_text).strip()
    machine_text = str(machine_text).strip()

    if human_text and human_status == "reviewed_approved":
        return human_text, "human"

    if (
        machine_text
        and machine_status == "machine_draft_generated"
    ):
        return machine_text, "machine"

    return "", "missing"


def resolve_translations_v2(df):

    result = df.copy()

    for index, row in result.iterrows():

        # Question selection
        question, question_source = select_translation_v2(
            row["edo_question_human"],
            row["edo_question_machine"],
            row["human_question_review_status"],
            row["machine_question_status"]
        )

        result.at[index, "edo_question_final"] = question
        result.at[
            index, "question_translation_selected_from"
        ] = question_source

        # Answer selection for agricultural Q&A only
        if row["translation_unit_type"] == "bilingual_qa":

            answer, answer_source = select_translation_v2(
                row["edo_answer_human"],
                row["edo_answer_machine"],
                row["human_answer_review_status"],
                row["machine_answer_status"]
            )

            result.at[index, "edo_answer_final"] = answer
            result.at[
                index, "answer_translation_selected_from"
            ] = answer_source

        else:
            result.at[index, "edo_answer_final"] = ""
            result.at[
                index, "answer_translation_selected_from"
            ] = "not_applicable_classifier_only"

    return result


translation_queue = resolve_translations_v2(
    machine_translation_df
)

print("Translation selection completed.")

Translation selection completed.


In [13]:
qa_mask = (
    translation_queue["translation_unit_type"]
    == "bilingual_qa"
)

outscope_mask = (
    translation_queue["translation_unit_type"]
    == "question_only_classifier"
)

coverage_summary = pd.DataFrame([
    {
        "translation_field": "Final Edo questions",
        "total_expected": 306,
        "available": translation_queue[
            "edo_question_final"
        ].ne("").sum()
    },
    {
        "translation_field": "Final Edo answers",
        "total_expected": 231,
        "available": translation_queue.loc[
            qa_mask, "edo_answer_final"
        ].ne("").sum()
    },
    {
        "translation_field": "Human Edo questions",
        "total_expected": 306,
        "available": translation_queue[
            "edo_question_human"
        ].ne("").sum()
    },
    {
        "translation_field": "Human Edo answers",
        "total_expected": 231,
        "available": translation_queue.loc[
            qa_mask, "edo_answer_human"
        ].ne("").sum()
    }
])

coverage_summary["missing"] = (
    coverage_summary["total_expected"]
    - coverage_summary["available"]
)

print("TRANSLATION COVERAGE")
display(coverage_summary)

print("\nQuestion selection:")
display(
    translation_queue[
        "question_translation_selected_from"
    ].value_counts()
)

print("\nAnswer selection:")
display(
    translation_queue[
        "answer_translation_selected_from"
    ].value_counts()
)

# Ensure uncertain answer is not selected
uncertain_answers = translation_queue[
    translation_queue["machine_answer_status"]
    == "translation_uncertain"
]

assert uncertain_answers["edo_answer_final"].eq("").all()
assert uncertain_answers[
    "answer_translation_selected_from"
].eq("missing").all()

# Ensure out-of-scope answers remain empty
assert translation_queue.loc[
    outscope_mask, "edo_answer_final"
].eq("").all()

print("\nTranslation coverage audit passed.")

TRANSLATION COVERAGE


,translation_field,total_expected,available,missing
0,Final Edo questions,306,306,0
1,Final Edo answers,231,230,1
2,Human Edo questions,306,0,306
3,Human Edo answers,231,0,231



Question selection:


question_translation_selected_from
machine    306
Name: count, dtype: int64


Answer selection:


answer_translation_selected_from
machine                           230
not_applicable_classifier_only     75
missing                             1
Name: count, dtype: int64


Translation coverage audit passed.


## 16. Exporting the Provisional Bilingual Dataset

We will export the current translation queue separately from the original English dataset.

The export retains:

- Original English questions and answers.
- Machine-generated Edo drafts.
- Human translation fields.
- Final selected translations.
- Translation selection source.
- Translation review and agricultural verification statuses.

This is a provisional bilingual dataset, not the final validated EdoFarm AI dataset.

It can be used for further development and testing, but machine-generated translations must remain identifiable as such.

In [14]:
provisional_path = (
    PROCESSED_DIR / "edofarm_bilingual_dataset_provisional_v1.csv"
)

coverage_path = (
    PROCESSED_DIR / "edofarm_translation_coverage_v1.csv"
)

# Export provisional bilingual dataset
translation_queue.to_csv(
    provisional_path,
    index=False,
    encoding="utf-8-sig"
)

coverage_summary.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig"
)

# Read back
provisional_export = pd.read_csv(
    provisional_path,
    dtype=str,
    keep_default_na=False
)

# Integrity checks
assert len(provisional_export) == 306
assert provisional_export["record_id"].is_unique

assert provisional_export[
    "edo_question_final"
].ne("").all()

assert provisional_export.loc[
    provisional_export["translation_unit_type"] == "question_only_classifier",
    "edo_answer_final"
].eq("").all()

assert provisional_export.loc[
    provisional_export["machine_answer_status"] == "translation_uncertain",
    "edo_answer_final"
].eq("").all()

print("PROVISIONAL BILINGUAL EXPORT COMPLETED")
print("---------------------------------------")
print("Total records:", len(provisional_export))
print("Agricultural Q&A:", qa_mask.sum())
print("Out-of-scope questions:", outscope_mask.sum())

print(
    "Selected machine questions:",
    (
        provisional_export[
            "question_translation_selected_from"
        ] == "machine"
    ).sum()
)

print(
    "Selected machine answers:",
    (
        provisional_export[
            "answer_translation_selected_from"
        ] == "machine"
    ).sum()
)

print(
    "Missing selected answers:",
    (
        provisional_export.loc[
            qa_mask, "edo_answer_final"
        ].eq("")
    ).sum()
)

print("\nSaved files:")
print(provisional_path)
print(coverage_path)

PROVISIONAL BILINGUAL EXPORT COMPLETED
---------------------------------------
Total records: 306
Agricultural Q&A: 231
Out-of-scope questions: 75
Selected machine questions: 306
Selected machine answers: 230
Missing selected answers: 1

Saved files:
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\edofarm_bilingual_dataset_provisional_v1.csv
c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed\edofarm_translation_coverage_v1.csv
